# Quantum Teleportation Lab
# Part 2b: Bob receives a message
## SIGGRAPH 2026, Los Angeles, CA, July 2026
## Andrew Glassner 

www.glassner.com

Copyright 2026 by Andrew Glassner. Licensed under CC BY-NC 4.0

This notebook and its associated notes can be found on https://github.com/blueberrymusic/Quantum-Teleportation

---

<div style="background-color: #f0fff0; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #2E8B57; margin-top: 0;">Bob's Infrastructure</h1></center>
<center><h3>Utility code</h3></center>
<ul>
<li><strong>Import libraries</strong></li>
<li><strong>make_Bobs_QC - build Bob's circuit</strong></li>
<li><strong>run_Bobs_QC - run Bob's circuit</strong></li>
<li><strong>execute_Bobs_circuit - combined build & run</li>
</ul
</div>

### Import all the packages we'll use

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister 
from qiskit.quantum_info import Statevector

import numpy as np
import matplotlib.pyplot as plt

# This is "magic" that lets Qiskit draw into notebook cells
%matplotlib inline

### Build Bob's circuit

In [ ]:
def make_Bobs_QC(bit_ms, bit_ma, qb_state, create_Bobs_gates):

    # Build Bob's circuit. We need three quantum bits, and because this
    # simulator supports measurement, we'll use one classical bit, too
    qs = QuantumRegister(1, 's')[0]   # Source qubit with state to teleport
    qa = QuantumRegister(1, 'a')[0]   # Alice's qubit
    qb = QuantumRegister(1, 'b')[0]   # Bob's qubit
    Bobs_QC = QuantumCircuit([qs, qa, qb])

    # Assign |0> and |1> to qubits qs and qa corresponding to the values
    # bit_ms and bit_ma that Alice measured (and told to us).
    
    # initialize() conveniently lets us specify a quantum state with a standard
    # numpy array of two (complex) numbers rather than a 2-by-1 matrix
    ket0 = np.array([1, 0])
    ket1 = np.array([0, 1])

    # Put either |0> or |1> into qs and qa, depending on their related bit
    Bobs_QC.initialize([ket0, ket1][bit_ms], qs)
    Bobs_QC.initialize([ket0, ket1][bit_ma], qa)

    # Initialize qb to the state that resulted from Alice's step
    Bobs_QC.initialize(qb_state, qb)   

    # Insert the gates of Bob's circuit    
    create_Bobs_gates(Bobs_QC, qs, qa, qb)

    return Bobs_QC

### Run Bob's circuit, return statistics of measuring 0

In [ ]:
def run_Bobs_QC(Bobs_QC):

    # Run Bob's circuit
    bottom_up_amplitudes = Statevector.from_instruction(Bobs_QC)

    # Qiskit reports amplitudes bottom-up, but I prefer top-down, so 
    # let's swap them around by reversing the bitstrings, e.g. state
    # 3 (011) becomes state 6 (110)
    amplitudes = bottom_up_amplitudes.data[[0, 4, 2, 6, 1, 5, 3, 7]]

    # Convert amplitudes to probabilities so we can simulate a measurement
    probabilities = np.abs(amplitudes)**2  

    # The probability of measuring 0 is the sum of probabilties of all states of the form xx0
    # That's 000=0, 010=2, 100=4, 110=6
    prob_alpha = probabilities[0] + probabilities[2] + probabilities[4] + probabilities[6]   
    
    # The probability of measuring 1 is the sum of probabilties of all states of the form xx1
    # That's 001=1, 011=3, 101=6, 111=7
    prob_beta  = probabilities[1] + probabilities[3] + probabilities[5] + probabilities[7]    

    # The amplitudes are the square roots of the probabilities
    qb_alpha = np.sqrt(prob_alpha)
    qb_beta = np.sqrt(prob_beta)

    return qb_alpha, qb_beta

### Perform Bob's step

In [ ]:
def execute_Bobs_circuit(bit_ms, bit_ma, filename, create_Bobs_gates):

    # Read the quantum state from the filename Alice provided
    qb_state = np.loadtxt(filename, dtype=complex)

    # Create Bob's quantum circuit using the classical bits Alice told us
    Bobs_QC = make_Bobs_QC(bit_ms, bit_ma, qb_state, create_Bobs_gates)
    
    alpha, beta = run_Bobs_QC(Bobs_QC)

    # We know alpha is a real number (because that's what Alice sent), so we can drop the complex part
    alpha = np.real(alpha)
    
    return Bobs_QC, alpha

<div style="background-color: #fff0f8; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #882E57; margin-top: 0;">Stop entering cells here</h1></center>
</div>

---

<div style="background-color: #f0f8ff; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #2E8B57; margin-top: 0;">Bob's Step</h1></center>
<center><h3>Recover the secret message</h3></center>
<ul>
<li><strong>Build Bob's circuit including final measurement</strong></li>
<li><strong>Initialize Bob's qubit and the two control qubits</strong></li>
<li><strong>Run it and extract the secret message</strong></li>
</ul>
</div>

### Bob Step 1: Provide the quantum gates that make up Bob's circuit, in order left to right

In [ ]:
def create_Bobs_gates(Bobs_QC, qs, qa, qb):   
    # Your code goes here

### Bob Step 2: Enter the bits and filename from Alice

In [ ]:
# Replace these lines with what Alice tells us
bit_ms = # Place Alice's measured value of s here
bit_ma = # Place Alice's measured value of a here
filename = 'filename' # Place the name of Alice's output file here

### Bob Step 3: Run Bob's circuit and recover the qubit

In [ ]:
Bobs_QC, alpha = execute_Bobs_circuit(bit_ms, bit_ma, filename, create_Bobs_gates)

# Draw Bob's circuit and report the recovered alpha amplitude of qb
Bobs_QC.draw('mpl')
print('received alpha = ',alpha)

---

# End of Part 2b: Quantum Teleportation